# Meeting Transcript & Summary Generator

Transcribe a meeting audio file (Thai / English) and produce an HTML summary.

**Pipeline**
1. Transcribe the meeting audio (Whisper, Thai + English aware).
2. Summarize the meeting into a styled HTML file (Google Gemini).
3. Save the result to a folder you choose (defaults to the machine's real Downloads folder).

Version: 2.2.0 · Date: 2026-08-06

## 1. Install dependencies

Run this cell once. It installs the transcription engine (`faster-whisper`), the Google Gemini
SDK (`google-genai`), and helpers. `ffmpeg` must also be available on the system PATH for most
audio formats (mp3, m4a, ...). See README for install instructions.

In [ ]:
# Transcript Generator - dependency installation
# Version: 2.0.0 | Date: 2026-08-06
# Installs the transcription engine, the Google Gemini SDK, and audio helpers.
%pip install -q faster-whisper google-genai python-dotenv

## 2. Imports & configuration

Detects the machine's **real** Downloads folder from the Windows registry, so a Downloads
folder relocated to another drive (e.g. `D:\Downloads`) is handled correctly — `Path.home()`
alone would wrongly point at `C:`. This value is only a suggested starting point; the actual
output folder is chosen explicitly in cell 3.

In [ ]:
# Imports and configuration
# Version: 2.1.0 | Date: 2026-08-06
import os
import sys
import html
import datetime as dt
from pathlib import Path

from dotenv import load_dotenv

# Load GEMINI_API_KEY from a local .env file if present (never commit .env).
load_dotenv()

# ---- User-configurable settings -------------------------------------------------

# Whisper model size: tiny | base | small | medium | large-v3
# 'medium' is a good Thai/English accuracy vs. speed balance on CPU.
MODEL_SIZE = "medium"

# Compute type: 'int8' is fast and light on CPU; use 'float16' on a CUDA GPU.
COMPUTE_TYPE = "int8"

# Language hint: 'th', 'en', or None to auto-detect per segment.
# None is recommended for mixed Thai/English meetings.
LANGUAGE_HINT = None

# Gemini model used for summarization. Aliases track the newest release so they
# don't break when Google retires a dated model name:
#   'gemini-flash-latest' = fast / cheap ; 'gemini-pro-latest' = best quality.
# If this name is unavailable on your account, the summary step auto-selects an
# available model and prints which one it used.
SUMMARY_MODEL = "gemini-flash-latest"

# Output language for the summary: 'th' (Thai) or 'en' (English).
SUMMARY_LANGUAGE = "th"


def get_downloads_dir() -> Path:
    """Return the machine's real Downloads folder.

    On Windows the Downloads folder can be relocated to another drive (e.g. D:),
    so Path.home()/'Downloads' is NOT reliable. Read the actual location from the
    'User Shell Folders' registry key, which honours a moved folder. Falls back to
    ~/Downloads on non-Windows systems or if the lookup fails.
    """
    if sys.platform.startswith("win"):
        try:
            import winreg

            key_path = r"Software\Microsoft\Windows\CurrentVersion\Explorer\Shell Folders"
            downloads_guid = "{374DE290-123F-4565-9164-39C4925E467B}"
            with winreg.OpenKey(winreg.HKEY_CURRENT_USER, key_path) as key:
                raw, _ = winreg.QueryValueEx(key, downloads_guid)
            # Value may contain env vars (e.g. %USERPROFILE%); expand them.
            return Path(os.path.expandvars(raw))
        except Exception as exc:
            print(f"Registry Downloads lookup failed ({exc}); using ~/Downloads.")

    return Path.home() / "Downloads"


# Suggested default only - the user is REQUIRED to confirm/choose in cell 3.
DEFAULT_DOWNLOADS = get_downloads_dir()

print(f"Whisper model      : {MODEL_SIZE} ({COMPUTE_TYPE})")
print(f"Summary model      : {SUMMARY_MODEL}")
print(f"Detected Downloads : {DEFAULT_DOWNLOADS}")

## 3. Choose the meeting audio file & output folder (required)

Running the next cell opens two native Windows dialogs: one to pick the meeting recording,
and one to pick the output folder (starting at the detected Downloads folder).

**Both choices are mandatory.** If a dialog is cancelled — or `tkinter` is unavailable — the
cell raises an error instead of guessing a path. This avoids silently writing to the wrong
drive when Downloads has been relocated. Supported audio: wav, mp3, m4a, flac, ogg, and more.

In [ ]:
# Select the meeting audio file and output folder via native dialogs (mandatory)
# Version: 1.2.0 | Date: 2026-08-06
#
# Selection is forced: there is no hard-coded path fallback. If a dialog is
# cancelled or tkinter is unavailable, the cell raises so nothing is ever saved
# to a guessed/wrong location (e.g. a relocated Downloads folder on another drive).


def _pick_paths():
    """Open Tk dialogs to choose the audio file and output folder.

    Returns (audio_path, output_dir). Raises RuntimeError if the dialog toolkit
    cannot be loaded; returns None for a value the user cancelled.
    """
    try:
        import tkinter as tk
        from tkinter import filedialog
    except Exception as exc:  # tkinter missing (rare headless build)
        raise RuntimeError(
            f"File dialogs unavailable ({exc}). Run this notebook on a desktop "
            "session, or set AUDIO_PATH/OUTPUT_DIR manually and skip this cell."
        ) from exc

    # A withdrawn root keeps the empty Tk window hidden; topmost forces focus.
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    audio = filedialog.askopenfilename(
        title="Select the meeting audio file",
        filetypes=[
            ("Audio files", "*.wav *.mp3 *.m4a *.flac *.ogg *.aac *.wma *.mp4"),
            ("All files", "*.*"),
        ],
    )

    # Only open the folder dialog if an audio file was actually chosen.
    out_dir = ""
    if audio:
        out_dir = filedialog.askdirectory(
            title="Select the OUTPUT folder for the HTML report",
            initialdir=str(DEFAULT_DOWNLOADS),
        )

    root.destroy()
    return (Path(audio) if audio else None), (Path(out_dir) if out_dir else None)


_audio, _out_dir = _pick_paths()

# Enforce mandatory selection - no fallback path.
if _audio is None:
    raise ValueError("No audio file selected. Re-run this cell and pick a file.")
if _out_dir is None:
    raise ValueError("No output folder selected. Re-run this cell and pick a folder.")

AUDIO_PATH = _audio
OUTPUT_DIR = _out_dir
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not AUDIO_PATH.exists():
    raise FileNotFoundError(f"Selected audio file does not exist: {AUDIO_PATH}")

print(f"Audio file    : {AUDIO_PATH}")
print(f"Output folder : {OUTPUT_DIR}")

## 4. Transcribe (Thai / English)

Loads the Whisper model once and transcribes the audio. Word-level timestamps are grouped
into readable segments. The first run downloads the model weights (cached afterwards).

In [ ]:
# Transcription with faster-whisper (Thai + English aware)
# Version: 1.0.0 | Date: 2026-08-06
from faster_whisper import WhisperModel


def format_timestamp(seconds: float) -> str:
    """Convert a float number of seconds to an HH:MM:SS string."""
    return str(dt.timedelta(seconds=int(seconds)))


def transcribe(audio_path: Path):
    """Transcribe an audio file and return (segments, info).

    segments: list of dicts with start, end, and text.
    info: detected language and probability.
    """
    # Load the model (CPU by default; set device='cuda' for a GPU).
    model = WhisperModel(MODEL_SIZE, device="cpu", compute_type=COMPUTE_TYPE)

    # beam_size=5 improves accuracy; vad_filter drops long silences.
    seg_iter, info = model.transcribe(
        str(audio_path),
        language=LANGUAGE_HINT,
        beam_size=5,
        vad_filter=True,
    )

    segments = []
    for seg in seg_iter:
        text = seg.text.strip()
        if not text:
            continue
        segments.append({"start": seg.start, "end": seg.end, "text": text})
        # Stream progress so a long meeting shows activity.
        print(f"[{format_timestamp(seg.start)}] {text}")

    return segments, info


segments, info = transcribe(AUDIO_PATH)
print(
    f"\nDetected language: {info.language} "
    f"(confidence {info.language_probability:.2f}) | {len(segments)} segments"
)

# Full transcript as a single string, one line per segment.
full_transcript = "\n".join(s["text"] for s in segments)

## 5. Summarize the meeting with Gemini

Sends the transcript to Google Gemini (`gemini-flash-latest`) and asks for a structured summary
in the chosen language.

The API key is read from `GEMINI_API_KEY` (in a `.env` file or the environment). If it isn't
set, a **popup window** asks for it (the input is masked). If no key is provided at all, a
simple offline fallback summary is produced instead.

In [ ]:
# Meeting summarization via the Google Gemini API
# Version: 2.1.0 | Date: 2026-08-06

SUMMARY_INSTRUCTIONS = {
    "th": (
        "คุณคือผู้ช่วยสรุปการประชุมมืออาชีพ. สรุปบันทึกการประชุมต่อไปนี้เป็นภาษาไทย "
        "โดยจัดรูปแบบเป็น Markdown และให้หัวข้อต่อไปนี้: บทสรุปสั้นๆ, หัวข้อหลักที่หารือ, "
        "การตัดสินใจ, และสิ่งที่ต้องทำต่อ (action items) พร้อมผู้รับผิดชอบหากระบุได้."
    ),
    "en": (
        "You are a meeting-notes assistant. Summarize the transcript below in English, "
        "formatted as Markdown. Provide these sections: an executive summary, key discussion "
        "points, decisions made, and action items with an owner where one is identifiable."
    ),
}


def get_gemini_api_key() -> str | None:
    """Return the Gemini API key.

    Resolution order:
      1. GEMINI_API_KEY / GOOGLE_API_KEY environment variable (or .env file).
      2. A masked popup window prompting the user to paste the key.
    Returns None if no key is available and the popup is cancelled/unavailable.
    """
    key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
    if key:
        return key.strip()

    # No key in the environment -> ask via a popup window.
    try:
        import tkinter as tk
        from tkinter import simpledialog
    except Exception as exc:  # tkinter missing (rare headless build)
        print(f"Key popup unavailable ({exc}); set GEMINI_API_KEY in .env instead.")
        return None

    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    # show='*' masks the key as it is typed.
    key = simpledialog.askstring(
        "Gemini API key",
        "Enter your Google Gemini API key:",
        show="*",
        parent=root,
    )
    root.destroy()
    return key.strip() if key else None


def resolve_model(client, preferred: str) -> str:
    """Return a model name that is actually usable on this account.

    Gemini retires dated model names over time (e.g. 'gemini-2.5-flash' becomes
    unavailable to new users). This queries the models the API key can call and
    returns `preferred` if available, otherwise the best flash-family fallback.
    """
    try:
        usable = [
            m.name.replace("models/", "")
            for m in client.models.list()
            if "generateContent" in (getattr(m, "supported_actions", None) or [])
        ]
    except Exception as exc:
        print(f"Could not list models ({exc}); trying '{preferred}' as-is.")
        return preferred

    if not usable:
        return preferred
    if preferred in usable:
        return preferred

    # Prefer a 'flash-latest' alias, then any flash model, then anything usable.
    for pick in (
        [m for m in usable if "flash" in m and "latest" in m]
        + [m for m in usable if "flash" in m]
        + usable
    ):
        print(f"Model '{preferred}' unavailable; using '{pick}'.")
        return pick
    return preferred


def summarize_with_gemini(transcript: str, language: str) -> str:
    """Summarize a transcript with Google Gemini. Returns Markdown-formatted text.

    Falls back to a naive local summary if no API key is provided.
    """
    api_key = get_gemini_api_key()
    if not api_key:
        print("No Gemini API key provided - using offline fallback summary.")
        return _fallback_summary(transcript, language)

    from google import genai
    from google.genai import types

    client = genai.Client(api_key=api_key)
    instructions = SUMMARY_INSTRUCTIONS.get(language, SUMMARY_INSTRUCTIONS["en"])
    model_name = resolve_model(client, SUMMARY_MODEL)

    response = client.models.generate_content(
        model=model_name,
        contents=transcript,
        config=types.GenerateContentConfig(system_instruction=instructions),
    )

    # response.text is None if the model returned no text (e.g. safety block).
    return response.text or _fallback_summary(transcript, language)


def _fallback_summary(transcript: str, language: str) -> str:
    """Offline summary used when no API key is available.

    Keeps the first lines as a rough preview so the pipeline still produces output.
    """
    lines = [ln for ln in transcript.splitlines() if ln.strip()]
    preview = "\n".join(f"- {ln}" for ln in lines[:15])
    header = (
        "## สรุปการประชุม (โหมดออฟไลน์)" if language == "th"
        else "## Meeting summary (offline mode)"
    )
    note = (
        "_ตั้งค่า GEMINI_API_KEY เพื่อสรุปด้วย Gemini ที่มีคุณภาพสูง._" if language == "th"
        else "_Set GEMINI_API_KEY for a high-quality Gemini summary._"
    )
    return f"{header}\n\n{note}\n\n{preview}"


summary_markdown = summarize_with_gemini(full_transcript, SUMMARY_LANGUAGE)
print(summary_markdown[:1000])

## 6. Build the HTML report

Renders the summary and the full timestamped transcript into a self-contained, printable
HTML document with a bright white theme and Thai-friendly fonts.

In [ ]:
# HTML report builder
# Version: 2.2.0 | Date: 2026-08-06


def markdown_to_html(text: str) -> str:
    """Minimal Markdown -> HTML for headings, bullets, and paragraphs.

    Intentionally dependency-free so the notebook stays lightweight.
    """
    out = []
    in_list = False
    for raw in text.splitlines():
        line = raw.rstrip()
        if not line:
            if in_list:
                out.append("</ul>")
                in_list = False
            continue
        if line.startswith("### "):
            if in_list:
                out.append("</ul>"); in_list = False
            out.append(f"<h3>{html.escape(line[4:])}</h3>")
        elif line.startswith("## "):
            if in_list:
                out.append("</ul>"); in_list = False
            out.append(f"<h2>{html.escape(line[3:])}</h2>")
        elif line.startswith("# "):
            if in_list:
                out.append("</ul>"); in_list = False
            out.append(f"<h2>{html.escape(line[2:])}</h2>")
        elif line.lstrip().startswith(("- ", "* ")):
            if not in_list:
                out.append("<ul>"); in_list = True
            out.append(f"<li>{html.escape(line.lstrip()[2:])}</li>")
        else:
            if in_list:
                out.append("</ul>"); in_list = False
            out.append(f"<p>{html.escape(line)}</p>")
    if in_list:
        out.append("</ul>")
    return "\n".join(out)


def build_html(title: str, summary_md: str, segments: list, info) -> str:
    """Assemble the final HTML document as a string (bright white theme)."""
    generated_at = dt.datetime.now().strftime("%Y-%m-%d %H:%M")
    summary_html = markdown_to_html(summary_md)

    rows = []
    for s in segments:
        ts = format_timestamp(s["start"])
        rows.append(
            f"<tr><td class='ts'>{ts}</td><td>{html.escape(s['text'])}</td></tr>"
        )
    transcript_rows = "\n".join(rows)

    # Self-contained document - no external assets, safe to open offline.
    # Forced light theme: 'color-scheme: light' + explicit colors keep the report
    # bright and white even when the OS/browser is set to dark mode.
    return f"""<!doctype html>
<html lang=\"th\">
<head>
<meta charset=\"utf-8\">
<meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">
<meta name=\"color-scheme\" content=\"light\">
<title>{html.escape(title)}</title>
<style>
  :root {{ color-scheme: light; }}
  html {{ background: #ffffff; }}
  body {{ font-family: 'Sarabun','Segoe UI',Tahoma,sans-serif; line-height:1.6;
         color:#1f2937; background:#ffffff;
         max-width: 900px; margin: 2rem auto; padding: 0 1.25rem; }}
  h1 {{ color:#111827; border-bottom: 3px solid #4f46e5; padding-bottom:.4rem; }}
  h2 {{ color:#4338ca; margin-top:2rem; }}
  h3 {{ color:#4338ca; }}
  a {{ color:#4f46e5; }}
  .meta {{ color:#6b7280; font-size:.9rem; }}
  section {{ background:#f8fafc; border:1px solid #e5e7eb; border-radius:12px;
            padding: .5rem 1.25rem 1.25rem; margin-top:1rem; }}
  ul {{ padding-left:1.25rem; }}
  table {{ border-collapse: collapse; width:100%; margin-top:1rem; background:#ffffff; }}
  td {{ border-bottom:1px solid #e5e7eb; padding:.45rem .6rem; vertical-align:top; }}
  tr:nth-child(even) td {{ background:#f9fafb; }}
  td.ts {{ white-space:nowrap; color:#4f46e5; font-variant-numeric:tabular-nums; width:5rem; }}
  details {{ margin-top:1.25rem; }}
  summary {{ cursor:pointer; font-weight:600; color:#4f46e5; }}
</style>
</head>
<body>
  <h1>{html.escape(title)}</h1>
  <p class=\"meta\">Generated {generated_at} · Detected language: {info.language} "
     f"({info.language_probability:.0%}) · {len(segments)} segments</p>

  <section>{summary_html}</section>

  <details>
    <summary>Full transcript / บันทึกการประชุมฉบับเต็ม</summary>
    <table>{transcript_rows}</table>
  </details>
</body>
</html>"""


report_title = f"Meeting Summary - {AUDIO_PATH.stem}"
html_document = build_html(report_title, summary_markdown, segments, info)
print(f"HTML built: {len(html_document):,} characters")

## 7. Save the report

Writes the report to the selected output folder (Downloads by default) with a timestamped
filename.

In [ ]:
# Save the HTML report to the selected output folder
# Version: 1.1.0 | Date: 2026-08-06

timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
safe_stem = "".join(c for c in AUDIO_PATH.stem if c.isalnum() or c in (" ", "_", "-")).strip()
output_file = OUTPUT_DIR / f"meeting_summary_{safe_stem}_{timestamp}.html"

output_file.write_text(html_document, encoding="utf-8")
print(f"Saved report to: {output_file}")